# Diamond Scraper — Luvansh + IGI Report Verification

This notebook:
1. Scrapes 10 round diamonds (1.8–2.5 ct) from [luvansh.com/shop-diamond](https://www.luvansh.com/shop-diamond)
2. Visits each diamond's detail page to find the IGI report number
3. Goes to the IGI Verify-Your-Report page and extracts proportions from the PDF Report tab
4. Filters diamonds matching ideal proportions:

| Parameter | Target Range |
|---|---|
| L/W Ratio | 1.00–1.02 |
| Table % | 54–58% |
| Depth % | 61.0–62.3% |
| Crown angle | 34.0–35.0° |
| Pavilion angle | 40.6–40.9° |
| Crown height % | 14.0–16.0% |
| Pavilion depth % | 42.5–43.2% |

## 1. Install Dependencies & Setup Chrome

In [ ]:
# Install required packages
# undetected-chromedriver is needed because both Luvansh and IGI use
# Cloudflare / bot-detection that blocks vanilla Selenium
!pip install -q undetected-chromedriver pdfplumber requests pandas
!apt-get update -qq
!apt-get install -qq -y chromium-browser chromium-chromedriver fonts-liberation

import sys, os
sys.path.insert(0, '/usr/lib/chromium-browser/chromedriver')
os.environ["DISPLAY"] = ":99"
print("Dependencies installed.")

## 2. Imports & Configuration

In [ ]:
import re
import time
import json
import io
import requests
import pdfplumber
import pandas as pd

# undetected-chromedriver bypasses Cloudflare & bot detection
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.common.exceptions import (
    TimeoutException, NoSuchElementException, StaleElementReferenceException
)

# ── Ideal-cut filter ranges ────────────────────────────────────────
IDEAL_RANGES = {
    "lw_ratio":       (1.00, 1.02),
    "table_pct":      (54.0, 58.0),
    "depth_pct":      (61.0, 62.3),
    "crown_angle":    (34.0, 35.0),
    "pavilion_angle": (40.6, 40.9),
    "crown_height":   (14.0, 16.0),
    "pavilion_depth": (42.5, 43.2),
}

# Number of diamonds to scrape per run
DIAMONDS_TO_SCRAPE = 10

# Carat range filter
CARAT_MIN = 1.8
CARAT_MAX = 2.5

# Shape filter
SHAPE = "Round"

print("Configuration loaded.")
print(f"Shape: {SHAPE}")
print(f"Carat range: {CARAT_MIN}–{CARAT_MAX}")
print(f"Diamonds to scrape: {DIAMONDS_TO_SCRAPE}")
print(f"Ideal ranges: {json.dumps({k: list(v) for k, v in IDEAL_RANGES.items()}, indent=2)}")

## 3. Selenium Helper — Create Headless Chrome Driver

In [ ]:
def create_driver():
    """
    Create an undetected-chromedriver instance for Google Colab.
    This bypasses Cloudflare and other bot-detection systems that
    block vanilla Selenium on both luvansh.com and igi.org.
    """
    options = uc.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")

    driver = uc.Chrome(options=options, version_main=None)
    driver.implicitly_wait(10)
    return driver

# Quick test
driver = create_driver()
print(f"Chrome driver ready — {driver.capabilities.get('browserVersion', 'unknown')}")
driver.quit()

## 4. Step 1 — Scrape Diamond Listings from Luvansh

In [ ]:
def scrape_luvansh_diamonds(driver, carat_min=1.8, carat_max=2.5, shape="Round", count=10):
    """
    Navigate to Luvansh shop-diamond page, apply filters via the website UI
    (shape, carat range), and return a list of diamond dicts.

    The Luvansh site has a filter panel on the left with:
      - Shape buttons (Round, Princess, Cushion, Oval, etc.)
      - Carat range slider / input fields
      - Color, Clarity, Cut checkboxes
    We interact with these filters using Selenium, then collect results.
    """
    base_url = "https://www.luvansh.com/shop-diamond"
    print(f"[Luvansh] Loading {base_url} ...")
    driver.get(base_url)
    time.sleep(6)  # Allow JS + React to fully render

    # ── Step A: Select shape (e.g. "Round") ───────────────────────
    print(f"[Luvansh] Selecting shape: {shape} ...")
    shape_clicked = False
    # Try clicking a shape button/label by text
    for selector in [
        f"//button[contains(translate(text(),'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'),'{shape.lower()}')]",
        f"//div[contains(@class,'shape')]//span[contains(translate(text(),'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'),'{shape.lower()}')]",
        f"//label[contains(translate(text(),'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'),'{shape.lower()}')]",
        f"//img[@alt='{shape}' or @alt='{shape.lower()}']/parent::*",
        f"//div[contains(@class,'filter')]//div[contains(translate(text(),'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'),'{shape.lower()}')]",
        f"//*[contains(@data-shape,'{shape.lower()}') or contains(@data-value,'{shape.lower()}')]",
    ]:
        try:
            el = driver.find_element(By.XPATH, selector)
            el.click()
            shape_clicked = True
            print(f"  [Filter] Shape '{shape}' selected via: {selector[:50]}...")
            time.sleep(2)
            break
        except (NoSuchElementException, Exception):
            continue

    if not shape_clicked:
        print(f"  [Filter] Could not click shape button — will try URL params")

    # ── Step B: Set carat range ───────────────────────────────────
    print(f"[Luvansh] Setting carat range: {carat_min}–{carat_max} ...")
    carat_set = False

    # Strategy 1: Find carat min/max input fields
    for sel_group in [
        ("input[placeholder*='Min' i]", "input[placeholder*='Max' i]"),
        ("input[placeholder*='From' i]", "input[placeholder*='To' i]"),
        ("input[name*='carat_min' i], input[name*='caratMin' i], input[name*='min_carat' i]",
         "input[name*='carat_max' i], input[name*='caratMax' i], input[name*='max_carat' i]"),
        ("input[id*='carat' i][id*='min' i]", "input[id*='carat' i][id*='max' i]"),
    ]:
        try:
            min_els = driver.find_elements(By.CSS_SELECTOR, sel_group[0])
            max_els = driver.find_elements(By.CSS_SELECTOR, sel_group[1])
            if min_els and max_els:
                min_el, max_el = min_els[0], max_els[0]
                # Clear and type using keyboard
                min_el.click()
                min_el.send_keys(Keys.CONTROL + "a")
                min_el.send_keys(str(carat_min))
                time.sleep(0.5)
                max_el.click()
                max_el.send_keys(Keys.CONTROL + "a")
                max_el.send_keys(str(carat_max))
                max_el.send_keys(Keys.RETURN)
                carat_set = True
                print(f"  [Filter] Carat range set via input fields")
                time.sleep(3)
                break
        except Exception:
            continue

    # Strategy 2: Look for a carat section with two generic inputs
    if not carat_set:
        try:
            carat_section = driver.find_elements(By.XPATH,
                "//*[contains(translate(text(),'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'),'carat')]"
                "/ancestor::div[contains(@class,'filter') or contains(@class,'range') or contains(@class,'section')]"
            )
            for section in carat_section:
                inputs = section.find_elements(By.TAG_NAME, "input")
                if len(inputs) >= 2:
                    inputs[0].clear()
                    inputs[0].send_keys(str(carat_min))
                    inputs[1].clear()
                    inputs[1].send_keys(str(carat_max))
                    inputs[1].send_keys(Keys.RETURN)
                    carat_set = True
                    print(f"  [Filter] Carat range set via section inputs")
                    time.sleep(3)
                    break
        except Exception:
            pass

    # Strategy 3: Try URL parameters as fallback
    if not carat_set or not shape_clicked:
        filter_url = (
            f"{base_url}?shape={shape}"
            f"&carat_min={carat_min}&carat_max={carat_max}"
            f"&caratmin={carat_min}&caratmax={carat_max}"
            f"&minCarat={carat_min}&maxCarat={carat_max}"
        )
        print(f"[Luvansh] Fallback — trying URL-based filter: {filter_url}")
        driver.get(filter_url)
        time.sleep(5)

    # ── Step C: Wait for results to load, then scroll to load more ─
    print("[Luvansh] Waiting for results to load ...")
    time.sleep(3)

    # Scroll down a few times to trigger lazy-loading
    for _ in range(3):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)

    # ── Step D: Collect diamond product links ─────────────────────
    diamonds = []
    product_links = driver.find_elements(By.CSS_SELECTOR, "a[href*='/product/']")

    # De-duplicate
    seen_urls = set()
    unique_links = []
    for link in product_links:
        try:
            href = link.get_attribute("href")
            if href and href not in seen_urls and "/product/" in href:
                seen_urls.add(href)
                unique_links.append((link, href))
        except StaleElementReferenceException:
            continue

    print(f"[Luvansh] Found {len(unique_links)} unique product links")

    for link_el, href in unique_links:
        if len(diamonds) >= count:
            break
        try:
            text = link_el.text.strip()

            # Extract carat from URL slug: "certified-2-03-e-vvs2" → 2.03
            carat_val = None
            url_match = re.search(r'certified-(\d+)-(\d+)-', href)
            if url_match:
                carat_val = float(f"{url_match.group(1)}.{url_match.group(2)}")

            # Also try from visible text
            if carat_val is None:
                carat_match = re.search(r'(\d+\.\d+)\s*(?:ct|carat)', text, re.I)
                if carat_match:
                    carat_val = float(carat_match.group(1))

            # Post-filter by carat if we couldn't use the website filter
            if carat_val is not None and not (carat_min <= carat_val <= carat_max):
                continue

            diamond = {
                "name": text or href.split("/")[-1].replace("-", " ").title(),
                "carat": carat_val,
                "detail_url": href,
            }
            diamonds.append(diamond)
        except (StaleElementReferenceException, Exception) as e:
            print(f"  [skip] Error reading link: {e}")
            continue

    diamonds = diamonds[:count]
    print(f"[Luvansh] Returning {len(diamonds)} diamonds (filtered to {carat_min}–{carat_max} ct)")
    return diamonds


# ── Run Step 1 ────────────────────────────────────────────────────
driver = create_driver()
try:
    diamonds = scrape_luvansh_diamonds(driver, CARAT_MIN, CARAT_MAX, SHAPE, DIAMONDS_TO_SCRAPE)
    for i, d in enumerate(diamonds, 1):
        print(f"  {i}. {d['name'][:60]}  |  {d.get('carat', '?')} ct  |  {d['detail_url']}")
finally:
    driver.quit()

## 5. Step 2 — Visit Each Diamond Detail Page → Extract IGI Report Number

In [ ]:
def extract_igi_report_number(driver, detail_url):
    """
    Visit a Luvansh diamond detail page and extract the IGI report number.
    Returns (report_number: str, carat_weight: float) or (None, None).
    """
    print(f"  [Detail] Loading {detail_url} ...")
    driver.get(detail_url)
    time.sleep(4)

    page_source = driver.page_source
    report_number = None
    carat_weight = None

    # ── Strategy 1: Find link to igi.org/Verify-Your-Report/?r=XXXXXX ──
    igi_links = driver.find_elements(By.CSS_SELECTOR, "a[href*='igi.org']")
    for link in igi_links:
        href = link.get_attribute("href") or ""
        m = re.search(r'[?&]r=(\d+)', href)
        if m:
            report_number = m.group(1)
            break

    # ── Strategy 2: Search page text for report patterns ──────────
    if not report_number:
        # IGI report numbers are typically 9-12 digits, often starting with LG
        patterns = [
            r'(?:IGI|Report|Certificate)\s*(?:#|No\.?|Number)?\s*:?\s*(?:LG)?(\d{7,12})',
            r'LG(\d{7,12})',
            r'Verify-Your-Report/\?r=(\d+)',
            r'report[_-]?(?:number|no|id)\s*[":]\s*["\']?(\d{7,12})',
        ]
        for pat in patterns:
            m = re.search(pat, page_source, re.I)
            if m:
                report_number = m.group(1)
                break

    # ── Strategy 3: Try extracting from the URL slug ──────────────
    if not report_number:
        # URL pattern: /product/2417100/...
        m = re.search(r'/product/(\d+)/', detail_url)
        if m:
            # The product ID might be the report number
            report_number = m.group(1)
            print(f"  [Detail] Using product ID as potential report number: {report_number}")

    # ── Extract carat weight from the page ────────────────────────
    carat_match = re.search(
        r'(?:Carat|Weight)\s*(?:Weight)?\s*:?\s*(\d+\.\d+)\s*(?:ct|carat|CT)',
        page_source, re.I
    )
    if carat_match:
        carat_weight = float(carat_match.group(1))

    # Fallback: extract from URL
    if carat_weight is None:
        url_m = re.search(r'certified-(\d+)-(\d+)-', detail_url)
        if url_m:
            carat_weight = float(f"{url_m.group(1)}.{url_m.group(2)}")

    # ── Also grab color, clarity, shape from the page ─────────────
    extra_info = {}
    color_m = re.search(r'Color\s*(?:Grade)?\s*:?\s*([A-Z])', page_source)
    if color_m:
        extra_info["color"] = color_m.group(1)

    clarity_m = re.search(r'Clarity\s*(?:Grade)?\s*:?\s*(\w+\s*\d*)', page_source)
    if clarity_m:
        extra_info["clarity"] = clarity_m.group(1).strip()

    # Try to find price
    price_m = re.search(r'[\$₹]\s*([\d,]+(?:\.\d{2})?)', page_source)
    if price_m:
        extra_info["price"] = price_m.group(0)

    if report_number:
        print(f"  [Detail] Found IGI report: {report_number}, carat: {carat_weight}")
    else:
        print(f"  [Detail] Could not find IGI report number")

    return report_number, carat_weight, extra_info


# ── Run Step 2 ────────────────────────────────────────────────────
driver = create_driver()
try:
    for d in diamonds:
        report_num, carat, extra = extract_igi_report_number(driver, d["detail_url"])
        d["igi_report_number"] = report_num
        if carat:
            d["carat"] = carat
        d.update(extra)
        time.sleep(2)  # Be polite
finally:
    driver.quit()

print("\n── Diamonds with IGI report numbers ──")
for i, d in enumerate(diamonds, 1):
    print(f"  {i}. Report={d.get('igi_report_number','N/A')}  Carat={d.get('carat','?')}  "
          f"Color={d.get('color','?')}  Clarity={d.get('clarity','?')}")

## 6. Step 3 — Go to IGI Verification → Extract Proportions

**Important notes from research:**
- IGI uses **Cloudflare bot protection** — `undetected-chromedriver` helps bypass this
- The verification page may have tabs: **Report Details**, **4C's**, **Journey**, **Videos**, and possibly **PDF Report**
- Some reports require **carat weight input** as an anti-scraping measure
- Only **full IGI reports** (not card-type certificates) contain proportions data
- The proportions diagram shows values like: `13.5% 58% 33.1° 40.9° 43% Pointed 61%`

In [ ]:
def extract_igi_proportions(driver, report_number, carat_weight=None):
    """
    Go to the IGI Verify-Your-Report page for the given report number,
    navigate to the tab with proportions data, and extract values.

    IGI verification page URL patterns:
      - https://www.igi.org/Verify-Your-Report/?r={report_number}
      - https://www.igi.org/verify.php?r={report_number}

    The page may have tabs: "Report Details", "PDF Report", "4C's", etc.
    Proportions are found in either the Report Details or PDF Report tab.

    Returns a dict with proportions data.
    """
    # Try the primary URL first, fallback to legacy
    urls_to_try = [
        f"https://www.igi.org/Verify-Your-Report/?r={report_number}",
        f"https://www.igi.org/verify.php?r={report_number}",
    ]

    for url in urls_to_try:
        print(f"  [IGI] Loading {url} ...")
        driver.get(url)
        time.sleep(6)  # Allow Cloudflare challenge + page load

        # Check if we got blocked
        page_text_lower = driver.page_source.lower()
        if "just a moment" in page_text_lower or "checking your browser" in page_text_lower:
            print("  [IGI] Cloudflare challenge detected — waiting 10s ...")
            time.sleep(10)

        # Check if page loaded successfully (has report data)
        if "report" in driver.page_source.lower() and "verify" in driver.page_source.lower():
            break

    # ── Handle potential carat weight verification form ────────────
    try:
        carat_input = driver.find_elements(By.CSS_SELECTOR,
            "input[name*='carat' i], input[placeholder*='carat' i], "
            "input[id*='carat' i], input[type='number']")
        if carat_input and carat_weight:
            carat_input[0].clear()
            carat_input[0].send_keys(str(carat_weight))
            # Look for a submit/check button
            submit_btns = driver.find_elements(By.CSS_SELECTOR,
                "button[type='submit'], input[type='submit'], "
                "button.btn-primary, button:not([disabled])")
            for btn in submit_btns:
                btn_text = btn.text.strip().lower()
                if any(w in btn_text for w in ["check", "verify", "submit", "search"]):
                    btn.click()
                    time.sleep(5)
                    break
            print(f"  [IGI] Submitted carat weight: {carat_weight}")
    except Exception as e:
        print(f"  [IGI] Carat verification step: {e}")

    # ── Try to click through all relevant tabs to find proportions ─
    # The page may have: "PDF Report" (id=pdf-tab), "Report Details", "4C's", etc.
    tab_selectors = [
        (By.ID, "pdf-tab"),
        (By.XPATH, "//button[contains(text(),'PDF')]"),
        (By.XPATH, "//a[contains(text(),'PDF')]"),
        (By.XPATH, "//button[contains(text(),'Report Details')]"),
        (By.XPATH, "//a[contains(text(),'Report Details')]"),
        (By.XPATH, "//button[contains(text(),'Report')]"),
        (By.XPATH, "//*[@role='tab']"),
    ]

    for by, sel in tab_selectors:
        try:
            tab = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((by, sel))
            )
            tab_text = tab.text.strip()
            tab.click()
            time.sleep(3)
            print(f"  [IGI] Clicked tab: '{tab_text}' ({sel})")
            break
        except (TimeoutException, NoSuchElementException):
            continue

    # ── Extract proportions from the page text + HTML ──────────────
    page_text = driver.find_element(By.TAG_NAME, "body").text
    page_source = driver.page_source
    combined = page_text + "\n" + page_source

    props = {
        "report_number": report_number,
        "measurements": None,
        "table_pct": None,
        "depth_pct": None,
        "crown_angle": None,
        "pavilion_angle": None,
        "crown_height": None,
        "pavilion_depth": None,
        "girdle": None,
        "culet": None,
        "polish": None,
        "symmetry": None,
        "fluorescence": None,
        "lw_ratio": None,
        "cut_grade": None,
        "carat_weight": None,
        "color_grade": None,
        "clarity_grade": None,
        "shape": None,
    }

    # ── Regex helper functions ────────────────────────────────────
    def find_float(pattern, text=combined):
        m = re.search(pattern, text, re.I)
        if m:
            try:
                return float(m.group(1))
            except ValueError:
                return None
        return None

    def find_str(pattern, text=combined):
        m = re.search(pattern, text, re.I)
        return m.group(1).strip() if m else None

    # ── Measurements: "8.13 - 8.16 X 4.96 MM" ────────────────────
    meas_m = re.search(
        r'Measurements?\s*:?\s*(\d+\.\d+\s*[-–]\s*\d+\.\d+\s*[Xx×]\s*\d+\.\d+)\s*(?:MM|mm)?',
        combined, re.I
    )
    if meas_m:
        props["measurements"] = meas_m.group(1).strip()
        dims = re.findall(r'(\d+\.\d+)', meas_m.group(1))
        if len(dims) >= 2:
            l, w = float(dims[0]), float(dims[1])
            if min(l, w) > 0:
                props["lw_ratio"] = round(max(l, w) / min(l, w), 3)

    # ── Named field extraction ────────────────────────────────────
    props["table_pct"] = find_float(r'Table\s*:?\s*(\d+(?:\.\d+)?)\s*%')
    props["depth_pct"] = find_float(r'Depth\s*:?\s*(\d+(?:\.\d+)?)\s*%')
    props["crown_angle"] = find_float(r'Crown\s*(?:Angle)?\s*:?\s*(\d+\.\d+)\s*[°]?')
    props["pavilion_angle"] = find_float(r'Pavilion\s*(?:Angle)?\s*:?\s*(\d+\.\d+)\s*[°]?')
    props["crown_height"] = find_float(r'Crown\s*(?:Height)?\s*(?:%?)?\s*:?\s*(\d+\.\d+)\s*%?')
    props["pavilion_depth"] = find_float(r'Pavilion\s*(?:Depth)?\s*(?:%?)?\s*:?\s*(\d+\.\d+)\s*%?')

    # ── IGI proportions diagram format (fallback) ─────────────────
    # The proportions diagram text typically appears as:
    #   "Medium To Slightly Thick (Faceted) 13.5% 58% 33.1° 40.9° 43% Pointed 61%"
    # Pattern: crown_height% table% crown_angle° pavilion_angle° pavilion_depth% culet depth%
    prop_pattern = re.search(
        r'(\d+\.\d+)%\s+(\d+)%\s+(\d+\.\d+)[°]\s+(\d+\.\d+)[°]\s+(\d+(?:\.\d+)?)%\s+\w+\s+(\d+(?:\.\d+)?)%',
        combined
    )
    if prop_pattern:
        print("  [IGI] Found proportions diagram pattern")
        if props["crown_height"] is None:
            props["crown_height"] = float(prop_pattern.group(1))
        if props["table_pct"] is None:
            props["table_pct"] = float(prop_pattern.group(2))
        if props["crown_angle"] is None:
            props["crown_angle"] = float(prop_pattern.group(3))
        if props["pavilion_angle"] is None:
            props["pavilion_angle"] = float(prop_pattern.group(4))
        if props["pavilion_depth"] is None:
            props["pavilion_depth"] = float(prop_pattern.group(5))
        if props["depth_pct"] is None:
            props["depth_pct"] = float(prop_pattern.group(6))

    # ── Also try: two angles next to each other ───────────────────
    if props["crown_angle"] is None or props["pavilion_angle"] is None:
        angles_m = re.search(r'(\d{2}\.\d+)[°]\s+(\d{2}\.\d+)[°]', combined)
        if angles_m:
            if props["crown_angle"] is None:
                props["crown_angle"] = float(angles_m.group(1))
            if props["pavilion_angle"] is None:
                props["pavilion_angle"] = float(angles_m.group(2))

    # ── Electronic copy format (bottom of IGI PDF) ────────────────
    # "61% 58% Medium To Slightly Thick (Faceted) Pointed EXCELLENT EXCELLENT NONE"
    # These are: Depth Table Girdle Culet Polish Symmetry Fluorescence
    ec_pattern = re.search(
        r'(\d+(?:\.\d+)?)%\s+(\d+(?:\.\d+)?)%\s+(\w[\w\s]*(?:\(Faceted\))?)\s+(Pointed|None|Very\s*Small|Small|Medium|Large)\s+'
        r'(EXCELLENT|VERY\s*GOOD|GOOD)\s+(EXCELLENT|VERY\s*GOOD|GOOD)\s+(NONE|FAINT|MEDIUM|STRONG)',
        combined, re.I
    )
    if ec_pattern:
        print("  [IGI] Found electronic copy pattern")
        if props["depth_pct"] is None:
            props["depth_pct"] = float(ec_pattern.group(1))
        if props["table_pct"] is None:
            props["table_pct"] = float(ec_pattern.group(2))
        if props["girdle"] is None:
            props["girdle"] = ec_pattern.group(3).strip()
        if props["culet"] is None:
            props["culet"] = ec_pattern.group(4).strip()
        if props["polish"] is None:
            props["polish"] = ec_pattern.group(5).strip()
        if props["symmetry"] is None:
            props["symmetry"] = ec_pattern.group(6).strip()
        if props["fluorescence"] is None:
            props["fluorescence"] = ec_pattern.group(7).strip()

    # ── Grading fields ────────────────────────────────────────────
    props["cut_grade"] = props["cut_grade"] or find_str(
        r'Cut\s*(?:Grade)?\s*:?\s*(IDEAL|EXCELLENT|VERY\s*GOOD|GOOD|FAIR|POOR)')
    props["polish"] = props["polish"] or find_str(
        r'Polish\s*:?\s*(EXCELLENT|VERY\s*GOOD|GOOD|FAIR|POOR)')
    props["symmetry"] = props["symmetry"] or find_str(
        r'Symmetry\s*:?\s*(EXCELLENT|VERY\s*GOOD|GOOD|FAIR|POOR)')
    props["fluorescence"] = props["fluorescence"] or find_str(
        r'Fluorescence\s*:?\s*(NONE|FAINT|MEDIUM|STRONG|VERY\s*STRONG)')
    props["girdle"] = props["girdle"] or find_str(
        r'Girdle\s*:?\s*([A-Za-z\s]+(?:\(Faceted\))?)')
    props["culet"] = props["culet"] or find_str(
        r'Culet\s*:?\s*(None|Pointed|Very\s*Small|Small|Medium|Large)')

    # Carat, Color, Clarity, Shape
    carat_v = find_float(r'Carat\s*Weight\s*:?\s*(\d+\.\d+)')
    if carat_v:
        props["carat_weight"] = carat_v
    props["color_grade"] = find_str(r'Color\s*Grade\s*:?\s*([A-Z])')
    props["clarity_grade"] = find_str(r'Clarity\s*Grade\s*:?\s*(\w+\s*\d*)')
    props["shape"] = find_str(
        r'(?:Shape|Cutting\s*Style)\s*:?\s*'
        r'(ROUND\s*BRILLIANT|ROUND|PRINCESS|CUSHION|OVAL|EMERALD|PEAR|MARQUISE|HEART|RADIANT)')

    # ── Try to find and parse an embedded PDF ─────────────────────
    try:
        pdf_elements = driver.find_elements(By.CSS_SELECTOR,
            "iframe[src*='.pdf'], embed[src*='.pdf'], object[data*='.pdf'], a[href*='.pdf']")
        for el in pdf_elements:
            pdf_url = el.get_attribute("src") or el.get_attribute("data") or el.get_attribute("href")
            if pdf_url:
                print(f"  [IGI] Found PDF URL: {pdf_url}")
                try:
                    pdf_props = extract_from_pdf(pdf_url)
                    for k, v in pdf_props.items():
                        if v is not None and props.get(k) is None:
                            props[k] = v
                except Exception as pdf_e:
                    print(f"  [IGI] PDF extraction failed: {pdf_e}")
                break
    except Exception:
        pass

    print(f"  [IGI] Extracted: Table={props['table_pct']}% Depth={props['depth_pct']}% "
          f"CrAngle={props['crown_angle']}° PavAngle={props['pavilion_angle']}° "
          f"CrHeight={props['crown_height']}% PavDepth={props['pavilion_depth']}% "
          f"L/W={props['lw_ratio']}")
    return props


def extract_from_pdf(pdf_url):
    """Download an IGI PDF report and extract proportions using pdfplumber."""
    props = {}
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"
    }
    resp = requests.get(pdf_url, headers=headers, timeout=30)
    resp.raise_for_status()

    with pdfplumber.open(io.BytesIO(resp.content)) as pdf:
        full_text = ""
        for page in pdf.pages:
            full_text += page.extract_text() or ""

    def find_float(pattern):
        m = re.search(pattern, full_text, re.I)
        return float(m.group(1)) if m else None

    props["table_pct"] = find_float(r'Table\s*:?\s*(\d+(?:\.\d+)?)\s*%?')
    props["depth_pct"] = find_float(r'Depth\s*:?\s*(\d+(?:\.\d+)?)\s*%?')
    props["crown_angle"] = find_float(r'Crown\s*(?:Angle)?\s*:?\s*(\d+\.\d+)\s*°?')
    props["pavilion_angle"] = find_float(r'Pavilion\s*(?:Angle)?\s*:?\s*(\d+\.\d+)\s*°?')
    props["crown_height"] = find_float(r'Crown\s*(?:Height)?\s*:?\s*(\d+\.\d+)\s*%?')
    props["pavilion_depth"] = find_float(r'Pavilion\s*(?:Depth)?\s*:?\s*(\d+\.\d+)\s*%?')

    # Proportions diagram pattern
    prop_m = re.search(
        r'(\d+\.\d+)%\s+(\d+)%\s+(\d+\.\d+)°\s+(\d+\.\d+)°\s+(\d+(?:\.\d+)?)%\s+\w+\s+(\d+(?:\.\d+)?)%',
        full_text
    )
    if prop_m:
        props.setdefault("crown_height", float(prop_m.group(1)))
        props.setdefault("table_pct", float(prop_m.group(2)))
        props.setdefault("crown_angle", float(prop_m.group(3)))
        props.setdefault("pavilion_angle", float(prop_m.group(4)))
        props.setdefault("pavilion_depth", float(prop_m.group(5)))
        props.setdefault("depth_pct", float(prop_m.group(6)))

    meas_m = re.search(r'(\d+\.\d+\s*[-–]\s*\d+\.\d+\s*[Xx×]\s*\d+\.\d+)', full_text)
    if meas_m:
        props["measurements"] = meas_m.group(1).strip()
        dims = re.findall(r'(\d+\.\d+)', meas_m.group(1))
        if len(dims) >= 2:
            l, w = float(dims[0]), float(dims[1])
            if min(l, w) > 0:
                props["lw_ratio"] = round(max(l, w) / min(l, w), 3)

    return props

print("IGI extraction functions defined.")

In [ ]:
# ── Run Step 3: Fetch IGI proportions for each diamond ────────────
driver = create_driver()
try:
    for d in diamonds:
        report_num = d.get("igi_report_number")
        if not report_num:
            print(f"  [SKIP] No IGI report number for: {d.get('name', '?')[:40]}")
            d["igi_props"] = {}
            continue

        carat = d.get("carat")
        props = extract_igi_proportions(driver, report_num, carat)
        d["igi_props"] = props

        # Merge carat if found from IGI
        if props.get("carat_weight") and not d.get("carat"):
            d["carat"] = props["carat_weight"]

        time.sleep(3)  # Be polite to IGI
finally:
    driver.quit()

print("\n── IGI data collection complete ──")
for i, d in enumerate(diamonds, 1):
    p = d.get("igi_props", {})
    print(f"{i}. Report={p.get('report_number','?')}  "
          f"Table={p.get('table_pct','?')}%  Depth={p.get('depth_pct','?')}%  "
          f"CrAngle={p.get('crown_angle','?')}°  PavAngle={p.get('pavilion_angle','?')}°  "
          f"L/W={p.get('lw_ratio','?')}")

## 7. Step 4 — Filter Diamonds by Ideal Proportions

In [ ]:
def check_ideal(props, ranges=IDEAL_RANGES):
    """
    Check which ideal-cut criteria a diamond meets.
    Returns (passes_all: bool, details: dict)
    """
    details = {}
    all_pass = True
    any_data = False

    for key, (lo, hi) in ranges.items():
        val = props.get(key)
        if val is None:
            details[key] = {"value": None, "pass": None, "range": f"{lo}–{hi}", "status": "NO DATA"}
            all_pass = False
            continue

        any_data = True
        in_range = lo <= val <= hi
        details[key] = {
            "value": val,
            "pass": in_range,
            "range": f"{lo}–{hi}",
            "status": "✓" if in_range else "✗"
        }
        if not in_range:
            all_pass = False

    return all_pass and any_data, details


# ── Apply filter ──────────────────────────────────────────────────
ideal_diamonds = []
near_ideal_diamonds = []

print("═" * 90)
print(f"{'#':>2}  {'Report':<14} {'Carat':>5}  {'Table%':>6}  {'Depth%':>6}  "
      f"{'CrAngle':>7}  {'PavAngle':>8}  {'CrHt%':>5}  {'PavDp%':>6}  {'L/W':>5}  {'Verdict'}")
print("═" * 90)

for i, d in enumerate(diamonds, 1):
    p = d.get("igi_props", {})
    passes, details = check_ideal(p)

    # Count how many criteria pass
    n_pass = sum(1 for v in details.values() if v["pass"] is True)
    n_total = len(IDEAL_RANGES)

    verdict = f"{n_pass}/{n_total}"
    if passes:
        verdict += " IDEAL"
        ideal_diamonds.append(d)
    elif n_pass >= n_total - 2:
        verdict += " NEAR"
        near_ideal_diamonds.append(d)

    print(f"{i:>2}  {p.get('report_number','?'):<14} {d.get('carat','?'):>5}  "
          f"{str(p.get('table_pct','?')):>6}  {str(p.get('depth_pct','?')):>6}  "
          f"{str(p.get('crown_angle','?')):>7}  {str(p.get('pavilion_angle','?')):>8}  "
          f"{str(p.get('crown_height','?')):>5}  {str(p.get('pavilion_depth','?')):>6}  "
          f"{str(p.get('lw_ratio','?')):>5}  {verdict}")

    # Print per-criterion details
    for key, info in details.items():
        if info["pass"] is False:
            print(f"      ✗ {key}: {info['value']} (need {info['range']})")

print("═" * 90)
print(f"\nResults: {len(ideal_diamonds)} IDEAL, {len(near_ideal_diamonds)} NEAR-IDEAL "
      f"out of {len(diamonds)} diamonds")

## 8. Summary Table (Pandas DataFrame)

In [ ]:
# ── Build a summary DataFrame ─────────────────────────────────────
rows = []
for d in diamonds:
    p = d.get("igi_props", {})
    passes, details = check_ideal(p)
    n_pass = sum(1 for v in details.values() if v["pass"] is True)

    rows.append({
        "IGI Report": p.get("report_number", ""),
        "Carat": d.get("carat"),
        "Color": p.get("color_grade") or d.get("color"),
        "Clarity": p.get("clarity_grade") or d.get("clarity"),
        "Cut": p.get("cut_grade"),
        "Shape": p.get("shape"),
        "Table %": p.get("table_pct"),
        "Depth %": p.get("depth_pct"),
        "Crown Angle": p.get("crown_angle"),
        "Pavilion Angle": p.get("pavilion_angle"),
        "Crown Height %": p.get("crown_height"),
        "Pavilion Depth %": p.get("pavilion_depth"),
        "L/W Ratio": p.get("lw_ratio"),
        "Polish": p.get("polish"),
        "Symmetry": p.get("symmetry"),
        "Fluorescence": p.get("fluorescence"),
        "Measurements": p.get("measurements"),
        "Criteria Met": f"{n_pass}/{len(IDEAL_RANGES)}",
        "Ideal?": "YES" if passes else "NO",
        "Luvansh URL": d.get("detail_url", ""),
        "Price": d.get("price", ""),
    })

df = pd.DataFrame(rows)

# Display with highlighting
def highlight_ideal(row):
    if row["Ideal?"] == "YES":
        return ["background-color: #c6efce"] * len(row)
    return [""] * len(row)

display(df.style.apply(highlight_ideal, axis=1).set_caption("Diamond Scraping Results"))

# Show only ideal diamonds
ideal_df = df[df["Ideal?"] == "YES"]
if not ideal_df.empty:
    print(f"\n{'='*60}")
    print(f"  IDEAL DIAMONDS FOUND: {len(ideal_df)}")
    print(f"{'='*60}")
    display(ideal_df[
        ["IGI Report", "Carat", "Color", "Clarity", "Table %", "Depth %",
         "Crown Angle", "Pavilion Angle", "Crown Height %", "Pavilion Depth %",
         "L/W Ratio", "Price", "Luvansh URL"]
    ])
else:
    print("\nNo diamonds met ALL ideal criteria in this batch.")
    print("Consider widening the ranges or scraping more diamonds.")

## 9. Export Results

In [ ]:
# Save to CSV
csv_filename = "diamond_results.csv"
df.to_csv(csv_filename, index=False)
print(f"Results saved to {csv_filename}")

# In Colab, offer download
try:
    from google.colab import files
    files.download(csv_filename)
except ImportError:
    print(f"(Not in Colab — file saved locally as {csv_filename})")

# Also save the raw data as JSON for debugging
json_filename = "diamond_results.json"
with open(json_filename, "w") as f:
    json.dump(diamonds, f, indent=2, default=str)
print(f"Raw data saved to {json_filename}")

## 10. Ideal Proportions Reference

| Parameter | Target Range | Why |
|---|---|---|
| L/W Ratio | 1.00–1.02 | Looks round; above ~1.03 starts to look slightly oval |
| Table % | 54–58% | Good balance between fire and sparkle |
| Depth % | 61.0–62.3% | Good light return without wasting diameter |
| Crown Angle | 34.0–35.0° | Crown angle that creates fire without hurting light return |
| Pavilion Angle | 40.6–40.9° | Critical for brilliance; outside range risks light leakage |
| Crown Height % | 14.0–16.0% | Usually matches a good crown angle |
| Pavilion Depth % | 42.5–43.2% | Optimal pavilion depth for light performance |